In [ ]:
# 1) locate the ablated weights (attached kernel output)
import os,subprocess
src=None
for r,d,f in os.walk("/kaggle/input"):
    if "model.safetensors" in f: src=r; break
print("weights at:",src)

In [ ]:
# 2) try direct HF->GGUF conversion (llama.cpp supports qwen3-vl text tower)
import subprocess
if not os.path.exists("/kaggle/working/llama.cpp"):
    subprocess.run(["git","clone","--depth","1","https://github.com/ggml-org/llama.cpp","/kaggle/working/llama.cpp"],check=True)
r=subprocess.run(["pip","install","-q","gguf","sentencepiece","protobuf"],capture_output=True,text=True)
print(r.stdout[-500:] or "deps ok", r.stderr[-300:] if r.returncode else "")
conv="/kaggle/working/llama.cpp/convert_hf_to_gguf.py"
r=subprocess.run(["python",conv,src,"--outfile","/kaggle/working/gremlin-f16.gguf","--outtype","f16"],capture_output=True,text=True)
print("convert rc:",r.returncode); print(r.stdout[-800:]); print(r.stderr[-800:])

In [ ]:
# 3) fallback: strip vision tower -> save text-only -> convert again
import os,subprocess,torch,json,glob
if not os.path.exists("/kaggle/working/gremlin-f16.gguf"):
    print("direct convert failed — stripping vision tower")
    from transformers import AutoModelForImageTextToText, AutoTokenizer
    m=AutoModelForImageTextToText.from_pretrained(src,torch_dtype=torch.float16)
    inner=m.model.language_model
    class Wrap(torch.nn.Module):
        def __init__(s,lm,head): super().__init__(); s.model=lm; s.lm_head=head
    w=Wrap(inner,m.lm_head)
    w.config=m.config.text_config
    os.makedirs("/kaggle/working/textonly",exist_ok=True)
    w.save_pretrained("/kaggle/working/textonly")
    AutoTokenizer.from_pretrained(src).save_pretrained("/kaggle/working/textonly")
    r=subprocess.run(["python","conv","/kaggle/working/textonly","--outfile","/kaggle/working/gremlin-f16.gguf","--outtype","f16"],capture_output=True,text=True)
    print("rc:",r.returncode,r.stderr[-600:])
else: print("f16 gguf already exists")

In [ ]:
# 4) build quantizer + make Q4_K_M
import subprocess,os
os.chdir("/kaggle/working/llama.cpp")
subprocess.run(["cmake","-B","build","-DGGML_NATIVE=ON"],capture_output=True,text=True)
subprocess.run(["cmake","--build","build","--target","llama-quantize","-j","4"],capture_output=True,text=True)
q=subprocess.run(["./build/bin/llama-quantize","/kaggle/working/gremlin-f16.gguf","/kaggle/working/gremlin-q4_k_m.gguf","Q4_K_M"],capture_output=True,text=True)
print(q.stdout[-600:]); print(q.stderr[-400:])
for f in os.listdir("/kaggle/working"):
    p=os.path.join("/kaggle/working",f)
    if os.path.isfile(p): print(f, round(os.path.getsize(p)/1e9,2),"GB")